# TieredKV: Sync 1 + Sync 2 GPU runs (Colab)

Runtime → Change runtime type → **GPU**. The free **T4 (15 GB) fits Llama-3.2-1B/3B, not 8B** (8B fp16 ≈ 16 GB; use an A100 for that).
Use the **same `MODEL` for every run** the team reports.

Order: check GPU → install → log in to Hugging Face → plan test 1 → full baseline (Sync 1) → train dictionary → lexico / tiered / h2o at 20% (Sync 2) → audit + plot → save results.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -b c-eval https://github.com/samarth2006-28/KV-cache.git
%cd KV-cache
!pip install -q -r requirements.txt
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())

Llama weights are gated: accept the licence on the model's Hugging Face page first, then paste **your own** access token when asked.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import os
MODEL = "meta-llama/Llama-3.2-1B-Instruct"   # same model for every run
N = 20                                        # raise once s/problem is known
DICT = "dicts/model_N4096_s16.pt"
os.environ["TIEREDKV_MODEL"] = MODEL

## Plan test 1: `full` cache == stock Hugging Face on 5 GSM8K prompts

In [ ]:
!python -m pytest -q tests/test_integration.py -k real_model

## Sync 1: full-cache baseline (upper bound) + seconds per problem

In [ ]:
!python -m tieredkv.eval --method full --budget 1.0 --n {N} --model {MODEL}

## Train the dictionary (Samarth's trainer; Lexico ships none)

In [ ]:
!mkdir -p dicts
!python -m tieredkv.train_dict --model {MODEL} --out {DICT} --minutes 20

## Sync 2: lexico vs tiered vs h2o at 20% memory

In [ ]:
!python -m tieredkv.eval --method lexico --budget 0.2 --n {N} --model {MODEL} --dictionary {DICT}
!python -m tieredkv.eval --method tiered --budget 0.2 --n {N} --model {MODEL} --dictionary {DICT}
!python -m tieredkv.eval --method h2o    --budget 0.2 --n {N} --model {MODEL}

## Budget audit (every row within 1 point of its target) + plot

In [ ]:
!python -m tieredkv.budget_audit results/results.csv --units fraction
!python plot.py
!cat results/results.csv

## Save results (Colab deletes files when the session ends)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/tieredkv && cp -r results dicts /content/drive/MyDrive/tieredkv/